### 크롤링으로 웹 페이지 정보를 수집하기전 주의사항
* `robots.txt` 파일 확인해서 수집을 허용하는지 확인
* 이용약관에서 자동 수집 금지, 상업적 이용 금지, 재배포 금지 조항을 확인
* 요청 간격을 적절히 지정

## 크롤링 과정
1. requests로 화면 html 코드를 가져옴
2. BS로 원하는 요소를 탐색
3. 탐색한 요소에 있는 정보를 추출
4. 활용(DataFrame)

### 1.requests로 화면 html 코드를 가져옴

In [6]:
import requests as req

BASE_URL = "https://kin.naver.com"


def fetch_kin_html(query:str, page:int=1) -> str | None:
    try:
        # params의 역할(get) : url 뒤에 ?변수=값형태로 변환
        # ?query=값&page=값
        result = req.get(f"{BASE_URL}/search/list.naver", params={"query":query, "page":page}, timeout=5)
        # print(result.url) # 확인용
        result.raise_for_status() # 정상응답이 아니면 예외를 발생시킴
        return result.text # 응답 결과에서 문자들을 반환(html 코드)
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None
html = fetch_kin_html("파이썬", 1)
# print(html.strip()) # 확인용

### 2.BS로 원하는 요소를 탐색
* 탐색하기전 원하는 요소의 선택자를 확인
* soup = BeautifulSoup(html코드, "html.parser") : html코드를 html객체로 분석
* els = soup.select("선택자") : 선택자와 일치하는 모든 요소들. 리스트로 반환
* el = soup.select_one("선택자") : 선택자와 일치하는 첫번째 요소.
* el.get_text(strip=True) : 공백을 제거하고 텍스트를 가져옴
* el.get("속성명"), el["속성명"] : 요소의 속성명과 일치하는 속성의 값을 가져옴

In [10]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser") #html코드로 된 문자열을 탐색이 쉽게 만들어줌
# 원하는 요소를 탐색
links = soup.select("._searchListTitleAnchor")
# 확인용
# for link in links:
#     print(link)

### 3. 탐색한 요소에 있는 정보를 추출

In [13]:
list = []
for link in links:
    # 제목
    title = link.get_text(strip=True)
    # 제목 링크
    href = link.get("href")
    list.append({"title" : title, "href" : href})

### 4. 활용

In [14]:
import pandas as pd
df = pd.DataFrame(list)
df.head(5)

,title,href
0,파이썬아이콘만드는방법,https://kin.naver.com/qna/detail.naver?dirId=1...
1,"파이썬기초 질문(urllib, beautifulsoup4)",https://kin.naver.com/qna/detail.naver?dirId=1...
2,파이썬리스트 슬라이싱,https://kin.naver.com/qna/detail.naver?dirId=1...
3,파이썬코딩 질문입니다.,https://kin.naver.com/qna/detail.naver?dirId=1...
4,정보 과목파이썬코드 질문,https://kin.naver.com/qna/detail.naver?dirId=1...


### 응용
* 검색어("파이선")으로 등록된 질문과 질문 상세, 첫번째 답변을 크롤링하는 작업
* df에 있는 href를 이용하여 질문 상세와 대표답변 1개를 가져오는 작업하세요
* 결과물은 df에 title, title_detail, answer가 들어가도록 작업

In [26]:
# 질문 링크를 이용하여 질문 상세 페이지 html 코드 가져오기
href = df.iloc[0]["href"]

# href를 이용해서 질문 상세 페이지에 있는 html 코드 가져오기
result = req.get(href, timeout=5)
html = result.text

In [18]:
# bs를 이용해서 요소를 탐색(질문 상세, 첫번째 답변)
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser") #html코드로 된 문자열을 탐색이 쉽게 만들어줌
# 원하는 요소를 탐색
detail_el = ""
answer_el = "" 
print(detail_el, answer_el)

In [ ]:
# 요소에서 필요한 내용 추출(질문 상세, 첫번째 답변)